# Method comparison -- in-silico detection and detection limit

Reads only `results/*.csv`; every pipeline / scenario written by `eval/build_results.sh <pipe>` shows up
on the next run. Methods on the x axis, evaluation scenario as grouped bars:

- `zero_shot`: no information from the held-out batch
- `adaptation`: half of the held-out batch's HC, cross-fitted -- mean-only shift for every method.
  CV has no adaptation (every batch is already in training).

Designs: in-silico sweep = `MethodComparison/sweep.py` (nb5), all held-out HC of each fold / batch;
detection limit = `3_detection_limit.ipynb`, half-boot of the held-out HC.

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

parent_dir = str(Path.cwd().parent)
if parent_dir not in sys.path:
    sys.path.insert(0, parent_dir)
from viz_style import FS_ANNOT_SM, apply_style
apply_style()

import _FinalModelingMethodImprovement.config as config

Q = 0.05
LOG2FC_SHOW = [0.0, 1.0, 2.0, 3.0, 5.0]
RATES = ['Precision', 'Recall', 'F1', 'FDR']
COUNTS = ['TP per sample', 'FP per sample', 'FN per sample']
METRICS = RATES + COUNTS
FIG_DIR = config.RESULTS_DIR / 'Figures'
FIG_DIR.mkdir(parents=True, exist_ok=True)

METHOD_ORDER = ['p0', 'p2', 'autoencoder', 'pca', 'outsingle', 'peer']
NAMES = {'p0': 'p0 current', 'p2': 'p2 RUV', **config.OUTLIER_ARMS}
COLORS = {'p0': '#000000', 'p2': '#0072B2', 'autoencoder': '#1B9E77',
          'pca': '#7570B3', 'outsingle': '#D95F02', 'peer': '#E7298A'}
SCENARIOS = ['zero_shot', 'adaptation']
HATCH = {'zero_shot': '', 'adaptation': '///'}
ALPHA = {'zero_shot': 1.0, 'adaptation': 0.45}
pd.set_option('display.width', 250)
pd.set_option('display.max_columns', 60)

In [ ]:
def summarize(df, unit):
    """Pool tp/fp/fn/tn over folds (or batch x boot) per split x method x scenario x q x log2fc.
    Counts per sample = pooled count / pooled number of scored samples."""
    keys = ['split', 'method', 'scenario', 'q', 'log2fc']
    g = df.groupby(keys)[['tp', 'fp', 'fn', 'tn']].sum()
    n = df.groupby(keys + unit)['n_samples'].first().groupby(keys).sum()
    tp, fp, fn = g['tp'], g['fp'], g['fn']
    return pd.DataFrame({'Precision': tp / (tp + fp), 'Recall': tp / (tp + fn),
                         'F1': 2 * tp / (2 * tp + fp + fn), 'FDR': fp / (tp + fp),
                         'TP per sample': tp / n, 'FP per sample': fp / n, 'FN per sample': fn / n}).reset_index()


def methods_in(s):
    return [m for m in METHOD_ORDER if m in set(s['method'])] + sorted(set(s['method']) - set(METHOD_ORDER))


def tables(s, title):
    """One table per split x metric: methods as rows, (log2fc, scenario) as columns."""
    s = s[s['q'] == Q]
    for split, sub in s.groupby('split'):
        print(f'===== {title} | {split} | q={Q}')
        for m in METRICS:
            t = sub[sub['log2fc'].isin(LOG2FC_SHOW)].pivot_table(index='method', columns=['log2fc', 'scenario'], values=m)
            print(f'--- {m}')
            display(t.reindex(methods_in(sub)).rename(index=NAMES).round(3 if m in RATES else 1))


def bars(ax, sub, value, methods):
    w = 0.8 / len(SCENARIOS)
    for k, scen in enumerate(SCENARIOS):
        v = sub[sub['scenario'] == scen].set_index('method')[value].reindex(methods)
        x = np.arange(len(methods)) + (k - (len(SCENARIOS) - 1) / 2) * w
        ax.bar(x, v.values, width=w, color=[COLORS.get(m, '#888888') for m in methods],
               alpha=ALPHA[scen], hatch=HATCH[scen], edgecolor='black', lw=0.4,
               label=scen.replace('_', '-'))
    ax.set_xticks(np.arange(len(methods)))
    ax.set_xticklabels([NAMES.get(m, m) for m in methods], rotation=45, ha='right')


def figure(s, title, fname):
    """Per split: rows = metrics (rates on 0-1, counts per sample on a linear axis shared
    along the row), columns = log2fc; x axes shared down each column."""
    s = s[s['q'] == Q]
    for split, sp in s.groupby('split'):
        methods = methods_in(sp)
        fig, axes = plt.subplots(len(METRICS), len(LOG2FC_SHOW), sharex=True, sharey='row',
                                 figsize=(2.9 * len(LOG2FC_SHOW), 1.7 * len(METRICS)), squeeze=False,
                                 gridspec_kw=dict(hspace=0.08, wspace=0.08))
        for j, fc in enumerate(LOG2FC_SHOW):
            for i, m in enumerate(METRICS):
                ax = axes[i, j]
                bars(ax, sp[sp['log2fc'] == fc], m, methods)
                if m in RATES:
                    ax.set_ylim(0, 1)
                if m == 'FDR':
                    ax.axhline(Q, color='red', ls=':', lw=0.8)
                if i == 0:
                    ax.set_title(f'log2FC = {fc:g}' + (' (no signal)' if fc == 0 else ''))
                if j == 0:
                    ax.set_ylabel(m)
        for ax in axes[:-1].flat:
            ax.tick_params(labelbottom=False)
        for ax in axes[:, 1:].flat:
            ax.tick_params(labelleft=False)
        axes[0, -1].legend(frameon=False, fontsize=FS_ANNOT_SM, loc='upper left', bbox_to_anchor=(1.02, 1))
        fig.suptitle(f'{title}: {split}, q = {Q}', y=0.995)
        fig.savefig(FIG_DIR / f'{fname}_{split}.png', dpi=300, bbox_inches='tight')
        plt.show()

## In-silico detection (sweep design)

In [ ]:
ins = summarize(pd.read_csv(config.INSILICO_CSV), unit=['fold'])
ins.to_csv(config.RESULTS_DIR / 'summary_insilico.csv', index=False)
print(ins.groupby(['split', 'method'])['scenario'].unique().to_string())
tables(ins, 'in-silico')

In [ ]:
figure(ins, 'In-silico', 'insilico')

## Detection limit (nb3 design)

In [ ]:
dl = summarize(pd.read_csv(config.DETECTION_LIMIT_CSV), unit=['batch', 'boot'])
dl.to_csv(config.RESULTS_DIR / 'summary_detection_limit.csv', index=False)
print(dl.groupby(['split', 'method'])['scenario'].unique().to_string())
tables(dl, 'detection limit')

In [ ]:
figure(dl, 'Detection limit', 'detection_limit')